# E-Commerce Exploratory Data Analysis (EDA)

**Objective:** Explore, clean, visualize and analyze the E-commerce dataset to identify patterns in customers, products, categories, sales, profit, discounts and payment modes.

This notebook keeps the original analysis and extends it with additional business-oriented EDA, outlier detection and final insights.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
# Keep the CSV in the same folder as this notebook
df = pd.read_csv("EDA_Ecommerce.csv")
df.head()

## 3. Basic Dataset Understanding

In [ ]:
print("Rows and Columns:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

## 4. Missing Values and Duplicate Records

In [ ]:
print("Missing values:
")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 5. Category and Brand Analysis

In [ ]:
print("Number of unique categories:", df["Category"].nunique())
print("\nCategory counts:")
print(df["Category"].value_counts())

print("\nTop brands:")
print(df["Brand"].value_counts().head(10))

In [ ]:
category_brand_map = df.groupby("Category")["Brand"].unique()
for category, brands in category_brand_map.items():
    print(category, ":", brands)

## 6. Data Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Fill missing values using simple, explainable rules
df["Age"] = df["Age"].fillna(df["Age"].mean())
df["City"] = df["City"].fillna("Unknown")
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].mean())
df["Payment_Mode"] = df["Payment_Mode"].fillna("Unknown")

print("Remaining missing values:
")
print(df.isnull().sum())

In [ ]:
# Check invalid customer ratings
invalid_rating = df[(df["Customer_Rating"] < 0) | (df["Customer_Rating"] > 5)]
print("Invalid customer rating records:", len(invalid_rating))
invalid_rating.head()

In [ ]:
# Treat invalid ratings as missing values for rating-based analysis
df.loc[(df["Customer_Rating"] < 0) | (df["Customer_Rating"] > 5), "Customer_Rating"] = np.nan
df["Customer_Rating"] = df["Customer_Rating"].fillna(df["Customer_Rating"].median())
print("Invalid ratings cleaned. Rating range:", df["Customer_Rating"].min(), "to", df["Customer_Rating"].max())

## 7. Univariate Analysis

In [ ]:
numerical_cols = ["Age", "Quantity", "Unit_Price", "Discount", "Customer_Rating", "Sales", "Profit"]
fig, axes = plt.subplots(4, 2, figsize=(12, 16))
axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
for j in range(len(numerical_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

## 8. Categorical Visualizations

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=df, x="Category", order=df["Category"].value_counts().index)
plt.xticks(rotation=45)
plt.title("Number of Orders by Category")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
top_brands = df["Brand"].value_counts().head(10)
sns.barplot(x=top_brands.values, y=top_brands.index)
plt.title("Top 10 Brands by Number of Orders")
plt.xlabel("Number of Orders")
plt.ylabel("Brand")
plt.tight_layout()
plt.show()

## 9. Bivariate Analysis

In [ ]:
corr = df[numerical_cols].corr()
display(corr)

plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 10. Sales and Profit Analysis

In [ ]:
category_sales_profit = df.groupby("Category")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False)
display(category_sales_profit)

In [ ]:
category_sales_profit.plot(kind="bar", figsize=(11,6))
plt.title("Sales and Profit by Category")
plt.xlabel("Category")
plt.ylabel("Amount")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Sales", y="Profit", hue="Category")
plt.title("Sales vs Profit")
plt.tight_layout()
plt.show()

## 11. Payment Mode Analysis

In [ ]:
payment_counts = df["Payment_Mode"].value_counts()
display(payment_counts)

plt.figure(figsize=(8,5))
sns.barplot(x=payment_counts.index, y=payment_counts.values)
plt.title("Orders by Payment Mode")
plt.xlabel("Payment Mode")
plt.ylabel("Number of Orders")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 12. City-wise Analysis

In [ ]:
city_sales = df.groupby("City")["Sales"].sum().sort_values(ascending=False).head(10)
display(city_sales)

plt.figure(figsize=(10,5))
sns.barplot(x=city_sales.values, y=city_sales.index)
plt.title("Top 10 Cities by Sales")
plt.xlabel("Sales")
plt.ylabel("City")
plt.tight_layout()
plt.show()

## 13. Customer Rating Analysis

In [ ]:
rating_counts = df["Customer_Rating"].round().value_counts().sort_index()
display(rating_counts)

plt.figure(figsize=(8,5))
sns.countplot(x=df["Customer_Rating"].round())
plt.title("Customer Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

## 14. Discount Analysis

In [ ]:
discount_summary = df.groupby("Discount")[["Sales", "Profit"]].mean().sort_index()
display(discount_summary)

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df, x="Discount", y="Profit")
plt.title("Discount vs Profit")
plt.tight_layout()
plt.show()

## 15. Quantity Analysis

In [ ]:
quantity_summary = df.groupby("Quantity")[["Sales", "Profit"]].sum().sort_values("Sales", ascending=False)
display(quantity_summary.head(10))

## 16. Top Products

In [ ]:
top_products_sales = df.groupby("Product_Name")["Sales"].sum().sort_values(ascending=False).head(10)
top_products_profit = df.groupby("Product_Name")["Profit"].sum().sort_values(ascending=False).head(10)

print("Top 10 Products by Sales")
display(top_products_sales)

print("Top 10 Products by Profit")
display(top_products_profit)

## 17. Outlier Detection using IQR

In [ ]:
outlier_summary = []
for col in ["Age", "Quantity", "Unit_Price", "Discount", "Customer_Rating", "Sales", "Profit"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append([col, q1, q3, lower, upper, count])

outlier_df = pd.DataFrame(outlier_summary, columns=["Column", "Q1", "Q3", "Lower Bound", "Upper Bound", "Outlier Count"])
display(outlier_df)

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df[["Age", "Quantity", "Unit_Price", "Discount", "Customer_Rating", "Sales", "Profit"]])
plt.title("Boxplots for Numerical Variables")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 18. Overall Business Summary

In [ ]:
print("===== E-COMMERCE EDA SUMMARY =====")
print("Total records:", len(df))
print("Total sales:", round(df["Sales"].sum(), 2))
print("Total profit:", round(df["Profit"].sum(), 2))
print("Average customer rating:", round(df["Customer_Rating"].mean(), 2))
print("Most frequent category:", df["Category"].value_counts().idxmax())
print("Top brand by order count:", df["Brand"].value_counts().idxmax())
print("Top product by sales:", df.groupby("Product_Name")["Sales"].sum().idxmax())
print("Top city by sales:", df.groupby("City")["Sales"].sum().idxmax())
print("Most used payment mode:", df["Payment_Mode"].value_counts().idxmax())

## 19. Key Insights and Conclusion

After running the notebook, use the generated tables and charts to describe: **(1)** which categories and products contribute most to sales, **(2)** which categories/products contribute most to profit, **(3)** the most common payment mode, **(4)** the strongest sales cities, **(5)** customer-rating patterns, **(6)** whether discounts appear to affect profit, and **(7)** which numerical columns contain important outliers.

### Conclusion
The E-commerce EDA project cleans the dataset, handles missing and duplicate records, checks data quality, explores numerical and categorical variables, studies relationships between important business measures, identifies top-performing categories/brands/products, and detects outliers. The final charts and summary values can be used to support data-driven business decisions.